# Synthetic Probe Generation — Method 3: Face Morphing

Pixel-level weighted blend of source and target face images after landmark alignment.
No generative model required — pure OpenCV.

```
morph = cv2.addWeighted(target_gallery_img, alpha, aligned_source, 1 - alpha, 0)
```

- **alpha = weight on target** (gallery image of true identity)
- **(1 - alpha) = weight on source** (probe from different identity)
- Faces are affine-aligned before blending to prevent ghosting

**Alphas generated**: 0.3, 0.5, 0.7 (primary analysis: 0.5)

This directly mirrors morphing attacks studied in passport/border control FR literature.
Unlike Methods 1–2, the entire image (background + face) is blended, so saliency maps
can reveal whether ArcFace is fooled by background leakage (Case B) vs genuine identity
features (Case A).

**Output**:
- `data/synthetic_probes/morphing/{identity}/{identity}_morph_{alpha}_{i}.jpg`
- Appends to `data/synthetic_probes/metadata.csv`

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

SPLIT_PATH      = "splits/lfw_1N_split.json"
GALLERY_EMB     = "cache/G.npy"
GALLERY_IDS     = "cache/gallery_ids.npy"
CRISE_SUMMARY   = "results/rise_alignedchip_baseline_multi/summary_K5_N1000_s8_p0.5_MASTERSEED123.csv"

OUT_DIR         = "data/synthetic_probes/morphing"
META_PATH       = "data/synthetic_probes/metadata.csv"
GEN_METHOD      = "morphing"

SYNTH_SEED      = 42        # same seed → same 50 identities + source pairings
N_TARGET_IDS    = 50
N_PROBES_PER_ID = 3

ALPHAS          = [0.3, 0.5, 0.7]   # alpha = weight on target; all three generated

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os, json
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from insightface.app import FaceAnalysis

from synth_gen import (
    face_morph_blend,
    get_embedding_from_image,
    select_target_identities,
    select_sources_for_target,
)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(META_PATH), exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup (detection + landmarks only; no swap model needed)
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("InsightFace ready")

In [ ]:
# ---------------------------------------------------------------------------
# Load split + gallery
# ---------------------------------------------------------------------------

with open(SPLIT_PATH) as f:
    split = json.load(f)

G           = np.load(GALLERY_EMB).astype(np.float32)
gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()
id_to_index = {gid: i for i, gid in enumerate(gallery_ids)}

print(f"Gallery: {G.shape[0]} identities")

In [ ]:
# ---------------------------------------------------------------------------
# Select 50 target identities  (same seed → same set as Methods 1 & 2)
# ---------------------------------------------------------------------------

rise_summary = pd.read_csv(CRISE_SUMMARY)
id_col = "true_id" if "true_id" in rise_summary.columns else rise_summary.columns[0]
completed_ids = rise_summary[id_col].dropna().unique().tolist()

target_ids = select_target_identities(completed_ids, n=N_TARGET_IDS, seed=SYNTH_SEED)
print(f"Target identities: {len(target_ids)} (seed={SYNTH_SEED})")
print(f"Alphas: {ALPHAS}  →  {len(target_ids) * N_PROBES_PER_ID * len(ALPHAS)} total probes")

In [ ]:
# ---------------------------------------------------------------------------
# Main generation loop — all alphas in one pass
# ---------------------------------------------------------------------------

records = []
n_ok = 0
n_fail = 0

for exp_i, target_id in enumerate(tqdm(target_ids, desc="Target identities")):
    target_gallery_path = split["gallery"][target_id]
    target_img = cv2.imread(target_gallery_path)
    if target_img is None:
        print(f"[WARN] Could not read gallery image for {target_id}")
        continue

    id_out_dir = os.path.join(OUT_DIR, target_id)
    os.makedirs(id_out_dir, exist_ok=True)

    source_pairs = select_sources_for_target(
        target_id, exp_i, split, n=N_PROBES_PER_ID, seed=SYNTH_SEED
    )

    for k, (src_id, src_path) in enumerate(source_pairs):
        source_img = cv2.imread(src_path)

        for alpha in ALPHAS:
            alpha_str = str(alpha).replace(".", "p")   # e.g. "0p5"

            if source_img is None:
                n_fail += 1
                records.append(dict(
                    identity=target_id, generation_method=GEN_METHOD,
                    source_identity=src_id, source_path=src_path,
                    blend_alpha=alpha, output_path=None, embedding_ok=False,
                    arcface_similarity=None, rank1_match=None,
                    saliency_cosine_sim=None, saliency_l1=None, case_label=None,
                ))
                continue

            morph = face_morph_blend(source_img, target_img, app, alpha=alpha)

            if morph is None:
                n_fail += 1
                records.append(dict(
                    identity=target_id, generation_method=GEN_METHOD,
                    source_identity=src_id, source_path=src_path,
                    blend_alpha=alpha, output_path=None, embedding_ok=False,
                    arcface_similarity=None, rank1_match=None,
                    saliency_cosine_sim=None, saliency_l1=None, case_label=None,
                ))
                continue

            out_fname = f"{target_id}_morph_{alpha_str}_{k}.jpg"
            out_path  = os.path.join(id_out_dir, out_fname)
            cv2.imwrite(out_path, morph, [cv2.IMWRITE_JPEG_QUALITY, 95])

            n_ok += 1
            records.append(dict(
                identity=target_id, generation_method=GEN_METHOD,
                source_identity=src_id, source_path=src_path,
                blend_alpha=alpha, output_path=out_path, embedding_ok=True,
                arcface_similarity=None, rank1_match=None,
                saliency_cosine_sim=None, saliency_l1=None, case_label=None,
            ))

print(f"\nGeneration complete: {n_ok} saved, {n_fail} failed")

In [ ]:
# ---------------------------------------------------------------------------
# Visualisation: one target identity, all 3 alphas side by side
# Rows = first 4 target identities;  Cols = source | α=0.3 | α=0.5 | α=0.7 | target
# ---------------------------------------------------------------------------

ok_records = [r for r in records if r["output_path"] is not None]
show_targets = target_ids[:4]

fig, axes = plt.subplots(len(show_targets), 5,
                         figsize=(13, 3 * len(show_targets)))

for row, tid in enumerate(show_targets):
    # one source per target (first k=0 pair)
    tid_recs = [r for r in ok_records if r["identity"] == tid]
    src_rec  = next((r for r in tid_recs if r["blend_alpha"] == 0.5), None)
    if src_rec is None:
        for col in range(5): axes[row, col].axis("off")
        continue

    src_img = cv2.cvtColor(cv2.imread(src_rec["source_path"]), cv2.COLOR_BGR2RGB)
    tgt_img = cv2.cvtColor(cv2.imread(split["gallery"][tid]),  cv2.COLOR_BGR2RGB)

    morph_imgs = {}
    for a in ALPHAS:
        r = next((r for r in tid_recs
                  if r["blend_alpha"] == a and r["source_identity"] == src_rec["source_identity"]
                  and r["output_path"] is not None), None)
        morph_imgs[a] = cv2.cvtColor(cv2.imread(r["output_path"]), cv2.COLOR_BGR2RGB) if r else None

    panels = [
        (src_img, f"Source\n{src_rec['source_identity'][:18]}"),
        (morph_imgs.get(0.3), "α=0.3\n(30% target)"),
        (morph_imgs.get(0.5), "α=0.5\n(50% target)"),
        (morph_imgs.get(0.7), "α=0.7\n(70% target)"),
        (tgt_img, f"Target (gallery)\n{tid[:18]}"),
    ]
    for col, (img, title) in enumerate(panels):
        if img is not None:
            axes[row, col].imshow(img)
        axes[row, col].set_title(title, fontsize=7)
        axes[row, col].axis("off")

plt.suptitle("Face Morphing — Source | α=0.3 | α=0.5 | α=0.7 | Target", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_grid.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Compute ArcFace embeddings for all generated morphs
# ---------------------------------------------------------------------------

synth_embeddings = {}

for r in tqdm(ok_records, desc="Embedding morphs"):
    emb = get_embedding_from_image(r["output_path"], app, rec)
    synth_embeddings[r["output_path"]] = emb
    if emb is None:
        r["embedding_ok"] = False

n_emb_ok   = sum(1 for e in synth_embeddings.values() if e is not None)
n_emb_fail = sum(1 for e in synth_embeddings.values() if e is None)
print(f"Embeddings: {n_emb_ok} OK, {n_emb_fail} failed")

In [ ]:
# ---------------------------------------------------------------------------
# Compute rank-1 match + cosine similarity to true identity
# ---------------------------------------------------------------------------

for r in records:
    if not r["embedding_ok"] or r["output_path"] is None:
        continue
    emb = synth_embeddings.get(r["output_path"])
    if emb is None:
        r["embedding_ok"] = False
        continue

    sims     = G @ emb
    true_idx = id_to_index[r["identity"]]
    r["arcface_similarity"] = round(float(sims[true_idx]), 6)
    r["rank1_match"]        = (int(np.argmax(sims)) == true_idx)

# Per-alpha breakdown
print("=== Rank-1 rate and mean similarity per alpha ===")
for a in ALPHAS:
    rows_a = [r for r in records if r["blend_alpha"] == a and r["rank1_match"] is not None]
    if not rows_a:
        continue
    r1 = sum(r["rank1_match"] for r in rows_a) / len(rows_a)
    ms = np.mean([r["arcface_similarity"] for r in rows_a])
    print(f"  alpha={a}:  rank-1={r1:.4f}  mean_sim={ms:.4f}  n={len(rows_a)}")

In [ ]:
# ---------------------------------------------------------------------------
# Append to shared metadata.csv (stale morphing rows dropped on re-run)
# ---------------------------------------------------------------------------

META_COLS = [
    "identity", "generation_method", "source_identity", "blend_alpha",
    "output_path", "embedding_ok", "arcface_similarity", "rank1_match",
    "saliency_cosine_sim", "saliency_l1", "case_label",
]

new_df = pd.DataFrame(records)[META_COLS]

if os.path.exists(META_PATH):
    existing = pd.read_csv(META_PATH)
    existing = existing[existing["generation_method"] != GEN_METHOD]
    combined = pd.concat([existing, new_df], ignore_index=True)
else:
    combined = new_df

combined.to_csv(META_PATH, index=False)
print(f"metadata.csv: {len(combined)} total rows → {META_PATH}")
print(combined.groupby("generation_method")[["embedding_ok", "rank1_match"]].mean().round(3))

In [ ]:
# ---------------------------------------------------------------------------
# Alpha ablation summary plot
# Shows how rank-1 rate and mean similarity vary with blend alpha
# ---------------------------------------------------------------------------

alpha_stats = []
for a in ALPHAS:
    rows_a = [r for r in records if r["blend_alpha"] == a and r["rank1_match"] is not None]
    if rows_a:
        alpha_stats.append(dict(
            alpha=a,
            rank1_rate=sum(r["rank1_match"] for r in rows_a) / len(rows_a),
            mean_sim=np.mean([r["arcface_similarity"] for r in rows_a]),
            std_sim=np.std([r["arcface_similarity"] for r in rows_a]),
        ))

if alpha_stats:
    df_alpha = pd.DataFrame(alpha_stats)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

    ax1.plot(df_alpha["alpha"], df_alpha["rank1_rate"], "o-", color="tab:blue")
    ax1.set_xlabel("alpha (weight on target)")
    ax1.set_ylabel("Rank-1 match rate")
    ax1.set_title("Fooling rate vs blend alpha")
    ax1.set_xticks(ALPHAS)
    ax1.grid(True, alpha=0.3)

    ax2.errorbar(df_alpha["alpha"], df_alpha["mean_sim"], yerr=df_alpha["std_sim"],
                 fmt="o-", color="tab:orange", capsize=4)
    ax2.set_xlabel("alpha (weight on target)")
    ax2.set_ylabel("Mean cosine sim to true identity")
    ax2.set_title("Similarity vs blend alpha")
    ax2.set_xticks(ALPHAS)
    ax2.grid(True, alpha=0.3)

    plt.suptitle("Face Morphing — Alpha Ablation", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "alpha_ablation.png"), dpi=150, bbox_inches="tight")
    plt.show()